# Optimisation Methods

This notebook defines the methods used in our experiments. Every method needs **NumPy** (`import numpy as np`). The other method-specific libraries are:

- **BO (`"BO"`)** uses `run_bo`. Install `scikit-optimize`; the function imports `skopt.gp_minimize` and `skopt.space.Real`.
- **CMA-ES (`"CMA-ES"`)** uses `run_cma_es`. Install `cma`; the function imports `cma`.
- **TuRBO-1 (`"TuRBO-1"`)** uses `run_turbo1`. Install `torch`, `gpytorch`, and Uber Research's TuRBO. The function imports `torch` and `turbo.Turbo1`.
- **Random (`"Random"`)** uses `run_uniform_random`.


## Calling a method

Every optimiser uses the same call structure:

```python
result = run_bo(objective, bounds, budget, seed, params)
```

Change the function name to `run_cma_es`, `run_turbo1`, or `run_uniform_random` to call another method. `run_method` at the end of this notebook selects a method by its experiment label.

- **`objective`**: two-coordinate point in the original domain and returns the objective value to minimise;
- **`bounds`** : `[lower, upper]` pair for each coordinate;
- **`budget`**: counts all objective evaluations;
- **`seed`** controls the method's random choices;
- **`params`** contains the method parameters used in `configurations.ipynb`. Random accepts an empty dictionary.

The following cell defines the shared evaluation recorder:

1. `EvaluationResults(objective, budget)` starts with empty point and a fixed budget.
2. Calling `recorder(point)` evaluates the objective once, appends the point to `X`, appends its value to `y`, and returns that value to the optimiser.
3. `recorder.result()` returns NumPy arrays for `X` and `y` and a `diagnostics` dictionary. 
`BudgetReached` is raised when the experimental budget is reached.

In [ ]:
import numpy as np


class BudgetReached(Exception):
    pass


class EvaluationResults:
    def __init__(self, objective, budget):
        self.objective, self.budget = objective, budget
        self.X, self.y = [], []

    def __call__(self, x):
        # CMA-ES generations and TuRBO restarts can otherwise exceed 50 calls.
        if len(self.y) == self.budget:
            raise BudgetReached
        point = np.array(x, dtype=float)
        value = float(self.objective(point))
        self.X.append(point)
        self.y.append(value)
        return value

    def result(self, **diagnostics):
        return {"X": np.asarray(self.X), "y": np.asarray(self.y),
                "diagnostics": diagnostics}

## Bayesian optimisation: `run_bo`

BO builds a Gaussian-process model of the objective from the values it has seen, then uses that model to choose the next evaluation. This implementation calls scikit-optimize's `gp_minimize`, with one continuous `Real` interval for each coordinate. It needs NumPy and **scikit-optimize**.

The current parameter dictionary contains four entries:

- **`n_initial_points`**: the number of random evaluations before the model begins (default initial-point generator: uniform random sampling)
- **`acq_func="EI"`** acquisition function selection.
- **`noise=1e-10`** observation-noise variance for numerical stability
- **`xi=0.0`** sets EI's improvement offset (Zero allows small improvements to contribute to the acquisition value, which matters near the precision targets).

```python
bo_params = {"n_initial_points": 4, "acq_func": "EI", "noise": 1e-10, "xi": 0.0}
result = run_bo(objective, bounds, budget=50, seed=42, params=bo_params)
```

Reference: [gp_minimize documentation](https://scikit-optimize.readthedocs.io/en/stable/modules/generated/skopt.gp_minimize.html).

In [ ]:
def run_bo(objective, bounds, budget, seed, params):
    from skopt import gp_minimize
    from skopt.space import Real

    recorder = EvaluationResults(objective, budget)
    gp_minimize(recorder, [Real(float(lo), float(hi)) for lo, hi in bounds],
                n_calls=budget, random_state=seed, **params)
    return recorder.result()

## CMA-ES

`run_cma_es` calls `cma.fmin2` in unit coordinates. CMA-ES samples a population around a mean and adapts its covariance and step size using the observed objective values. Unlike BO, it does not fit a global surrogate model.

The original domain is transformed coordinate by coordinate using

$$x = \mathrm{lower} + (\mathrm{upper}-\mathrm{lower})u, \qquad u\in[0,1]^2.$$

The following cell draws each initial mean uniformly in this unit square and records objective evaluations back in the original domain. **`sigma0=0.25`** is the initial step size in unit coordinates, corresponding to 2 units per coordinate in the original width-8 domain. The current experiment uses PyCMA's default population size.

```python
cma_params = {"sigma0": 0.25}
result = run_cma_es(objective, bounds, budget=50, seed=42, params=cma_params)
```

If PyCMA terminates before exhausting the budget, the loop draws a new initial mean and spends the remaining calls. The first internal start uses `seed + 1`; each subsequent start increments that internal seed. The separate NumPy generator controls the sequence of initial means. 

For an experiment that supplies an evaluated initial design, pass its best point as `initial_mean` in the original domain coordinates. This sets the first population mean. The budget passed to `run_cma_es` counts the new evaluations after the initial design. If the optimiser restarts, the next mean is drawn uniformly as usual.

In [ ]:
def run_cma_es(objective, bounds, budget, seed, params, *, initial_mean=None):
    import cma

    bounds = np.asarray(bounds)
    lower, upper = bounds.T
    width = upper - lower
    recorder = EvaluationResults(objective, budget)
    rng = np.random.default_rng(seed)
    starts = 0

    while len(recorder.y) < budget:
        options = {
            "bounds": [0.0, 1.0],
            "maxfevals": budget - len(recorder.y),
            "seed": seed + starts + 1,
            "eval_final_mean": False,
            "verbose": -9,
            "verb_log": 0,
        }

        if starts == 0 and initial_mean is not None:
            mean = (np.asarray(initial_mean) - lower) / width
        else:
            mean = rng.uniform(size=len(bounds))

        starts += 1
        try:
            cma.fmin2(
                lambda x: recorder(lower + width * x),
                mean,
                params["sigma0"],
                options=options,
            )
        except BudgetReached:
            pass

    return recorder.result(restarts=starts - 1)

## TuRBO-1: `run_turbo1`

TuRBO-1 builds a Gaussian-process model near a promising point and searches a trust region around it. That region expands after progress and shrinks when progress stalls. It can restart when the region becomes too small. This function uses Uber Research's implementation and needs NumPy, **PyTorch**, **GPyTorch**, and **TuRBO**.

The current parameter dictionary contains **`n_initial_points=4`**, which sets the number of initial design points for each trust-region start. The implementation's existing defaults supply the rest of the behaviour used by the experiment:

- **Batch size 1:** evaluations are proposed sequentially.
- **ARD enabled:** the model can learn a separate length scale for each coordinate.
- **50 training steps:** the Gaussian process is fitted using the default training schedule.
- **CPU and float64:** the model runs on the CPU in double precision.

```python
turbo_params = {"n_initial_points": 4}
result = run_turbo1(objective, bounds, budget=50, seed=42, params=turbo_params)
```
Reference: [Turbo1 implementation](https://github.com/uber-research/TuRBO/blob/master/turbo/turbo_1.py).

In [ ]:
def run_turbo1(objective, bounds, budget, seed, params):
    import torch
    from turbo import Turbo1

    bounds = np.asarray(bounds)
    recorder = EvaluationResults(objective, budget)
    np.random.seed(seed)
    torch.manual_seed(seed)
    optimiser = Turbo1(recorder, bounds[:, 0], bounds[:, 1],
                       n_init=params["n_initial_points"], max_evals=budget, verbose=False)
    try:
        optimiser.optimize()
    except BudgetReached:
        pass
    return recorder.result()

## Uniform random search: `run_uniform_random`

Random search draws each point uniformly and independently from the bounds. It does not use earlier values to choose later points. This is the simplest baseline and needs **NumPy only**.

It takes the same shared inputs, with an empty parameter dictionary:

```python
result = run_uniform_random(objective, bounds, budget=50, seed=42, params={})
```

`params` is accepted to keep consistency, but Random does not read it.


In [ ]:
def run_uniform_random(objective, bounds, budget, seed, params):
    lower, upper = np.asarray(bounds).T
    recorder = EvaluationResults(objective, budget)
    rng = np.random.default_rng(seed)
    for _ in range(budget):
        recorder(rng.uniform(lower, upper))
    return recorder.result()

## Choosing a method by name: `run_method`

`run_method(name, objective, bounds, budget, seed, params)` chooses one of the four functions above and passes inputs to it. The selections are `"BO"`, `"CMA-ES"`, `"TuRBO-1"`, and `"Random"`. 

In [ ]:
def run_method(name, objective, bounds, budget, seed, params):
    methods = {"BO": run_bo, "CMA-ES": run_cma_es,
               "TuRBO-1": run_turbo1, "Random": run_uniform_random}
    return methods[name](objective, bounds, budget, seed, params)